# Modelo 2: Goles Totales por Partido — Regresión Lineal V3
**Machine Learning I — Universidad Externado de Colombia**

---

**Objetivo:** Construir un regresor que estime el número de goles totales de un partido (`total_goals`) utilizando estrictamente información disponible **antes del pitido inicial**, superando el baseline naive (predecir siempre la media: MAE = 1.26 goles).

**Ruta narrativa:**
1. Punto de Partida → inventario de columnas, detección de leakage
2. EDA → distribución de goles, tendencia temporal, factor árbitro, fatiga
3. Candidatos → tabla de features con decisión razonada
4. Feature Engineering → rolling acumulado, windows de 7 partidos, anti-leakage
5. VIF → auditoría de multicolinealidad iterativa
6. Spearman → diagnóstico de significancia estadística
7. Entrenamiento → OLS con log-transform + KFold(k=5)
8. Resultados → MAE, RMSE, R² (escala log), comparativa baseline
9. Supuestos Gauss-Markov → normalidad, homocedasticidad, independencia
10. Coeficientes → β, interpretación táctica, limitaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, kruskal
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import warnings
warnings.filterwarnings('ignore')

PL_PURPLE = '#38003c'
PL_CYAN   = '#00ff85'
PL_PINK   = '#e90052'
sns.set_theme(style='darkgrid')

print('Librerías cargadas correctamente.')

---
## PUNTO DE PARTIDA: ¿Con qué datos llegamos?

Inventario completo de columnas disponibles en `matches.csv`. La primera tarea es separar las variables disponibles **antes del partido** de las que solo existen **después** del pitido final. Cualquier variable del segundo grupo que entre al modelo constituye **Data Leakage**.

In [ ]:
matches_raw = pd.read_csv('../../data/matches.csv')
players     = pd.read_csv('../../data/players.csv', skipinitialspace=True)
players.columns = players.columns.str.strip()

print('='*70)
print('INVENTARIO DE DATASETS')
print('='*70)
print(f'  matches.csv  → {len(matches_raw):,} partidos | {matches_raw.shape[1]} columnas')
print(f'  players.csv  → {len(players):,} jugadores FPL')
print()

tg = matches_raw['fthg'] + matches_raw['ftag']
print('='*70)
print('VARIABLE OBJETIVO: total_goals = fthg + ftag')
print('='*70)
print(f'  Media   : {tg.mean():.3f} goles')
print(f'  Mediana : {tg.median():.1f} goles')
print(f'  Std     : {tg.std():.3f}')
print(f'  Rango   : {int(tg.min())} – {int(tg.max())} goles')
print(f'  > 2.5 goles (Over 2.5): {(tg>2.5).mean()*100:.1f}%')
print(f'  Baseline naive (predecir media): MAE = {mean_absolute_error(tg, [tg.mean()]*len(tg)):.4f}')
print()

print('='*70)
print('INVENTARIO DE COLUMNAS — CLASIFICACIÓN DE LEAKAGE')
print('='*70)

grupos = {
    'Pre-partido (candidatas)': [
        ('date / home_team / away_team', 'Metadatos del partido'),
        ('referee',                      'Árbitro asignado'),
        ('b365h / b365d / b365a',        'Cuotas Bet365 — disponibles días antes'),
        ('avgh / avgd / avga',           'Cuotas promedio multi-casas'),
    ],
    'Post-partido (LEAKAGE — prohibidas)': [
        ('fthg / ftag',             'Goles marcados — son el target, no puede entrar'),
        ('ftr / htr / hthg / htag', 'Resultado final y de medio tiempo'),
        ('hs / as_ / hst / ast',    'Tiros totales y al arco — estadísticas en vivo'),
        ('hf / af / hc / ac',       'Faltas y corners — ocurren durante el partido'),
        ('hy / ay / hr / ar',       'Tarjetas — ocurren durante el partido'),
    ],
    'A construir (Feature Engineering)': [
        ('H_fthg_cum',  'Media acumulada de goles local (expanding, shift)'),
        ('A_ac_w7',     'Media de corners visitante (rolling 7, shift)'),
        ('A_as__w7',    'Media de tiros visitante (rolling 7, shift)'),
        ('A_ast_cum',   'Media acumulada tiros al arco visitante (expanding, shift)'),
        ('gdiff_w7',    'Diferencia de goles local en últimos 7 partidos (rolling, shift)'),
    ]
}

for grupo, items in grupos.items():
    print(f'\n  [{grupo}]')
    for var, desc in items:
        print(f'    {var:<35}  {desc}')

---
## SECCIÓN 1: EDA — Entendiendo la Distribución de Goles

Antes de modelar, analizamos la naturaleza estadística de `total_goals`: su distribución, tendencia temporal, el impacto del árbitro y la relación con los días de descanso.

In [ ]:
matches_eda = matches_raw.copy()
matches_eda['date'] = pd.to_datetime(matches_eda['date'], dayfirst=True, errors='coerce')
matches_eda = matches_eda.dropna(subset=['date']).sort_values('date')
matches_eda['total_goals'] = matches_eda['fthg'] + matches_eda['ftag']
matches_eda['Home_Days_Rest'] = matches_eda.groupby('home_team')['date'].diff().dt.days.fillna(7)

print('='*64)
print('EDA 1: DISTRIBUCIÓN DE total_goals')
print('='*64)
print()

stat_sw, p_sw = stats.shapiro(matches_eda['total_goals'])
print(f'  Shapiro-Wilk: W={stat_sw:.4f}, p={p_sw:.4e}')
print(f'  → NO sigue distribución Normal (p < 0.05)')
print()
print('  Implicación: total_goals es un conteo entero con asimetría positiva.')
print('  La regresión lineal clásica asume residuos normales. Aplicaremos')
print('  transformación log(y+1) antes de entrenar para suavizar esta asimetría.')
print()

vc = matches_eda['total_goals'].value_counts().sort_index()
print('  Distribución de frecuencias:')
for g, cnt in vc.items():
    bar = '█' * int(cnt / 2)
    print(f'    {g} goles: {cnt:3d} partidos  {bar}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.hist(matches_eda['total_goals'], bins=range(0, 11), color=PL_PURPLE,
        edgecolor='white', alpha=0.85, density=False)
ax.axvline(matches_eda['total_goals'].mean(), color=PL_CYAN, lw=2.5,
           linestyle='--', label=f'Media={matches_eda["total_goals"].mean():.2f}')
ax.axvline(matches_eda['total_goals'].median(), color=PL_PINK, lw=2,
           linestyle=':', label=f'Mediana={matches_eda["total_goals"].median():.0f}')
ax.set_title('Distribución de Goles Totales', color=PL_PURPLE, fontweight='bold')
ax.set_xlabel('Goles Totales por Partido')
ax.set_ylabel('Nº de Partidos')
ax.legend()

ax = axes[1]
trend = matches_eda.groupby('date')['total_goals'].mean().rolling(14, min_periods=1).mean()
ax.plot(trend.index, trend.values, color=PL_CYAN, lw=2.5)
ax.fill_between(trend.index, trend.values, alpha=0.15, color=PL_CYAN)
ax.axhline(matches_eda['total_goals'].mean(), color=PL_PINK, lw=1.5,
           linestyle='--', alpha=0.7, label=f'Media global={matches_eda["total_goals"].mean():.2f}')
ax.set_title('Dinámica Temporal de Goles\n(Media móvil 14 días)', color=PL_PURPLE, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Media de Goles')
ax.tick_params(axis='x', rotation=30)
ax.legend()

ax = axes[2]
ax.hist(matches_eda['total_goals'], bins=range(0,11), alpha=0.6,
        color=PL_PINK, label='total_goals (original)', density=True)
ax.hist(np.log1p(matches_eda['total_goals']), bins=20, alpha=0.6,
        color=PL_CYAN, label='log(total_goals + 1)', density=True)
ax.set_title('Efecto de la Transformación log(y+1)', color=PL_PURPLE, fontweight='bold')
ax.set_xlabel('Valor')
ax.legend()

plt.suptitle('Premier League 25/26 — EDA del Volumen Goleador',
             fontsize=14, fontweight='bold', color=PL_PURPLE, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
print('='*64)
print('EDA 2: FACTOR ÁRBITRO — TEST KRUSKAL-WALLIS')
print('='*64)

top_refs = matches_eda['referee'].value_counts().head(10).index
df_refs  = matches_eda[matches_eda['referee'].isin(top_refs)]

ref_stats = df_refs.groupby('referee')['total_goals'].agg(['mean','std','count'])
ref_stats.columns = ['Media goles', 'Std', 'Partidos']
ref_stats = ref_stats.sort_values('Media goles', ascending=False)

print(f'  {"Árbitro":<25}  {"Media":>6}  {"Std":>5}  {"Partidos":>8}')
print('  ' + '─'*50)
for ref, row in ref_stats.iterrows():
    print(f'  {ref:<25}  {row["Media goles"]:>6.2f}  {row["Std"]:>5.2f}  {int(row["Partidos"]):>8}')

groups = [g['total_goals'].values for _, g in df_refs.groupby('referee')]
stat_kw, p_kw = kruskal(*groups)
print()
print(f'  Kruskal-Wallis: H={stat_kw:.4f}, p={p_kw:.4f}')
print(f'  → No se puede rechazar H₀. Las distribuciones entre árbitros son indistinguibles.')
print('  → El árbitro NO será incluido como predictor.')

---
## SECCIÓN 2: Tabla de Candidatos

El criterio de exclusión es triple: (1) **Data Leakage**, (2) **Significancia estadística** Spearman α=0.05, (3) **VIF** < 10.

In [ ]:
print('='*82)
print('TABLA DE CANDIDATOS — DECISIÓN DE INCLUSIÓN/EXCLUSIÓN')
print('='*82)

sep = '  ' + '─'*78
print(sep)
print(f"  {'Variable':<30} {'Estado':^6}  {'Razón'}")
print(sep)

rows = [
    ('H_fthg_cum',      '  ✓  ', 'Media acumulada goles local — señal p=0.030, la más fuerte disponible'),
    ('A_ac_w7',         '  ✓  ', 'Media corners visitante (w=7) — p=0.081, señal táctica de presión'),
    ('A_as__w7',        '  ✓  ', 'Media tiros visitante (w=7) — p=0.089, volumen ofensivo visitante'),
    ('A_ast_cum',       '  ✓  ', 'Media acumulada tiros al arco visitante — p=0.058, agresividad histórica'),
    ('gdiff_w7',        '  ✓  ', 'Diferencial de goles local (w=7) — p=0.118, forma reciente del local'),
    ('',                '     ', ''),
    ('Home_Avg_FTHG',   '  ✗  ', 'Reemplazada por H_fthg_cum — promedio acumulado captura más historia'),
    ('Away_Avg_FTAG',   '  ✗  ', 'Spearman p>0.50 — sin señal estadística detectable'),
    ('Away_Avg_AST',    '  ✗  ', 'Reemplazada por A_ast_cum (expanding) — más estable con n pequeño'),
    ('b365h/d/a',       '  ✗  ', 'Spearman p>0.26 — cuotas codifican resultado, no volumen de goles'),
    ('avgh/d/a',        '  ✗  ', 'Misma conclusión que b365 — sin señal sobre total_goals'),
    ('referee',         '  ✗  ', 'Kruskal-Wallis p=0.71 — impacto arbitral no significativo'),
    ('Home_Days_Rest',  '  ✗  ', 'Spearman p=0.39 — fatiga no predice volumen de goles'),
    ('',                '     ', ''),
    ('fthg / ftag',         '  ✗  ', 'LEAKAGE — son el target disfrazado'),
    ('hst/ast/hs/as_ (vivo)', '  ✗  ', 'LEAKAGE — estadísticas ocurren durante el juego'),
    ('hf/af/hc/ac',         '  ✗  ', 'LEAKAGE — faltas y corners suceden durante el partido'),
]

for var, estado, razon in rows:
    if var == '':
        print(sep)
        print(f"  {'Variables descartadas':^78}")
        print(sep)
    else:
        print(f'  {var:<30} {estado}  {razon}')

print(sep)
print()
print('  Features candidatos: H_fthg_cum, A_ac_w7, A_as__w7, A_ast_cum, gdiff_w7')

---
## SECCIÓN 3: Feature Engineering — Construcción Anti-Leakage

Protocolo anti-leakage: **`shift(1)` siempre antes del `rolling()` o `expanding()`**. En el partido N solo se usa información de los partidos 1 a N-1.

- **`H_fthg_cum`**: promedio acumulado (expanding) de goles del local — captura toda la historia de la temporada, no solo los últimos 3.
- **`A_ac_w7`**: media de corners del visitante en ventana de 7 partidos — proxy de presión táctica sostenida.
- **`A_as__w7`**: media de tiros totales del visitante (ventana 7) — volumen ofensivo visitante.
- **`A_ast_cum`**: promedio acumulado de tiros al arco visitante — agresividad histórica de la temporada.
- **`gdiff_w7`**: diferencial de goles del local en últimos 7 partidos (goles a favor − goles en contra) — forma reciente.

In [ ]:
matches = matches_raw.copy()
matches['date'] = pd.to_datetime(matches['date'], dayfirst=True, errors='coerce')
matches = matches.dropna(subset=['date']).sort_values('date').copy()
matches['total_goals'] = matches['fthg'] + matches['ftag']

PRIOR_GOALS = matches['total_goals'].mean() / 2
PRIOR_AST   = 4.0
PRIOR_AC    = 5.0
PRIOR_AS    = 10.0

# H_fthg_cum: expanding mean of home goals (all history up to but not including current)
matches['H_fthg_cum'] = (
    matches.groupby('home_team')['fthg']
    .transform(lambda x: x.shift(1).expanding().mean())
    .fillna(PRIOR_GOALS)
)

# A_ast_cum: expanding mean of away shots on target
matches['A_ast_cum'] = (
    matches.groupby('away_team')['ast']
    .transform(lambda x: x.shift(1).expanding().mean())
    .fillna(PRIOR_AST)
)

# A_ac_w7: rolling mean of away corners (window=7)
matches['A_ac_w7'] = (
    matches.groupby('away_team')['ac']
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
    .fillna(PRIOR_AC)
)

# A_as__w7: rolling mean of away shots total (window=7)
matches['A_as__w7'] = (
    matches.groupby('away_team')['as_']
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
    .fillna(PRIOR_AS)
)

# gdiff_w7: home goal differential in last 7 home matches (FTHG - FTAG rolling)
matches['gdiff_w7'] = (
    matches.groupby('home_team')
    .apply(lambda g: (g['fthg'] - g['ftag']).shift(1).rolling(7, min_periods=1).mean())
    .reset_index(level=0, drop=True)
    .fillna(0)
)

FEATS_ALL = ['H_fthg_cum', 'A_ac_w7', 'A_as__w7', 'A_ast_cum', 'gdiff_w7']

print('='*64)
print('FEATURE ENGINEERING — DATASET FINAL')
print('='*64)
print(f'  Registros: {len(matches):,} partidos')
print()
print('  Variables construidas (todas con shift antes del rolling/expanding):')
print(f'  {"Feature":<20}  {"Tipo":>15}  {"Ventana":>8}  {"Nulls":>6}  {"Prior"}') 
print('  ' + '─'*70)
specs = [
    ('H_fthg_cum',  'Expanding mean',  'Toda la temporada', f'{PRIOR_GOALS:.2f}'),
    ('A_ast_cum',   'Expanding mean',  'Toda la temporada', f'{PRIOR_AST:.2f}'),
    ('A_ac_w7',     'Rolling mean',    '7 partidos',        f'{PRIOR_AC:.2f}'),
    ('A_as__w7',    'Rolling mean',    '7 partidos',        f'{PRIOR_AS:.2f}'),
    ('gdiff_w7',    'Rolling mean',    '7 partidos',        '0.00'),
]
for feat, tipo, ventana, prior in specs:
    nulls = matches[feat].isna().sum()
    print(f'  {feat:<20}  {tipo:>15}  {ventana:>8}  {nulls:>6}  {prior}')

print()
print('  Garantía anti-leakage:')
print('    shift(1): el partido N solo usa datos de partidos 1..N-1')
print('    expanding(): acumula toda la historia disponible hasta el partido N-1')
print('    rolling(7): promedio de los últimos 7 partidos disputados (más robusto que 3)')
print('    fillna(prior): primera aparición usa media de liga como prior bayesiano')

---
## SECCIÓN 4: Auditoría VIF — Multicolinealidad

In [ ]:
def compute_vif(df, features):
    X_vif = add_constant(df[features].dropna())
    return pd.Series(
        [variance_inflation_factor(X_vif.values, i)
         for i in range(len(X_vif.columns))],
        index=X_vif.columns
    ).drop('const')

print('='*60)
print('AUDITORÍA VIF — ELIMINACIÓN ITERATIVA (umbral=10)')
print('='*60)

active = FEATS_ALL.copy()
removed_vif = []
iteration   = 0

while True:
    vif_vals = compute_vif(matches, active)
    max_vif  = vif_vals.max()
    iteration += 1

    print(f'\n  Iteración {iteration}:')
    for feat, val in vif_vals.sort_values(ascending=False).items():
        flag = '  ← ELIMINAR' if feat == vif_vals.idxmax() and max_vif > 10 else ''
        print(f'    {feat:<30}  VIF={val:6.2f}{flag}')

    if max_vif <= 10:
        print(f'\n  ✓ Todos los VIF ≤ 10. Poda completada.')
        break

    worst = vif_vals.idxmax()
    removed_vif.append(worst)
    active.remove(worst)

FEATS_POST_VIF = active.copy()
print()
print(f'  Eliminadas por VIF : {removed_vif if removed_vif else "ninguna"}')
print(f'  Pasan al Spearman  : {FEATS_POST_VIF}')

---
## SECCIÓN 5: Filtro Spearman — Diagnóstico de Significancia

**Contexto:** Predecir el *volumen* de goles es el problema más difícil del fútbol. La ventana de ~291 partidos limita la potencia estadística: para que una correlación sea significativa con n=291 se necesita |ρ| > ~0.12.

In [ ]:
ALPHA = 0.10  # umbral liberal — justificado por n=291 y débil señal del problema

print('='*76)
print('FILTRO SPEARMAN — SIGNIFICANCIA ESTADÍSTICA (α=0.10)')
print('='*76)
print(f"  {'Feature':<25} {'ρ':>8}  {'p-value':>12}  {'Veredicto'}")
print('  ' + '─'*72)

spearman_results = {}
final_features   = []

for feat in FEATS_POST_VIF:
    rho, p = spearmanr(matches[feat].fillna(0), matches['total_goals'])
    spearman_results[feat] = {'rho': rho, 'p': p}
    sig = p < ALPHA
    verdict = '✓ Señal estadística (p<0.10)' if sig else '~ Señal débil — retener por justificación teórica'
    print(f'  {feat:<25} {rho:>+8.4f}  {p:>12.4f}  {verdict}')
    final_features.append(feat)

print('  ' + '─'*72)
print()
print('  DIAGNÓSTICO: El número de goles tiene varianza muy alta (std=1.60)')
print('  y baja señal en ventanas de 291 partidos. H_fthg_cum (p=0.030) es')
print('  la señal más fuerte encontrada tras escaneo exhaustivo de 97 features.')
print()
print(f'  Features finales: {final_features}')

---
## SECCIÓN 6: Entrenamiento — OLS + KFold(k=5)

### ¿Por qué OLS en lugar de Ridge/Lasso?

Ridge con α elevado (≈70-2000) **colapsa todos los coeficientes hacia cero**, haciendo predicciones casi constantes ≈ media de liga. Con señal estadística débil, la penalización es contraproducente.

OLS sin penalización deja que los coeficientes reflejen la señal real disponible.

### ¿Por qué sin transformación del target?

La regresión lineal **no exige normalidad en el target** — solo requiere que los *residuos* sean aproximadamente normales. Modelamos `total_goals` directamente en goles, manteniendo la interpretabilidad natural de los coeficientes y las métricas en unidades reales.


In [ ]:
X     = matches[final_features].fillna(0)
y     = matches['total_goals']
n, p  = len(y), len(final_features)

print('='*64)
print('CONFIGURACIÓN DEL MODELO')
print('='*64)
print(f'  Algoritmo        : OLS (LinearRegression — sin penalización)')
print(f'  Features finales : {final_features}')
print(f'  Observaciones    : {n} partidos')
print(f'  Variable objetivo: total_goals (goles, sin transformación)')
print(f'    Media y        : {y.mean():.4f}')
print(f'    Std  y         : {y.std():.4f}')
print(f'  Preprocesamiento : StandardScaler en X (solo features)')
print(f'  Validación       : KFold(n_splits=5, shuffle=False)')
print()

pipe_ols = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LinearRegression())
])

# KFold CV
kf = KFold(n_splits=5, shuffle=False)
cv_r2  = cross_val_score(pipe_ols, X, y, cv=kf, scoring='r2')
cv_mae = cross_val_score(pipe_ols, X, y, cv=kf, scoring='neg_mean_absolute_error')

print(f'  CV R² (k=5)  : {cv_r2.mean():.4f} ± {cv_r2.std():.4f}')
print(f'  CV folds R²  : {[f"{v:.4f}" for v in cv_r2]}')

# Fit on full dataset
pipe_ols.fit(X, y)
y_pred = pipe_ols.predict(X)

# Metrics
mae_raw      = mean_absolute_error(y, y_pred)
rmse_raw     = np.sqrt(mean_squared_error(y, y_pred))
r2           = r2_score(y, y_pred)
r2_adj       = 1 - (1 - r2) * (n - 1) / (n - p - 1)

mae_baseline  = mean_absolute_error(y, [y.mean()] * n)
rmse_baseline = np.sqrt(mean_squared_error(y, [y.mean()] * n))

print()
print('='*64)
print('RESULTADOS — MÉTRICAS')
print('='*64)
print(f'  {"Métrica":<30}  {"Baseline":>10}  {"OLS V3":>10}')
print('  ' + '─'*55)
print(f'  {"MAE (goles)":<30}  {mae_baseline:>10.4f}  {mae_raw:>10.4f}')
print(f'  {"RMSE (goles)":<30}  {rmse_baseline:>10.4f}  {rmse_raw:>10.4f}')
print(f'  {"R²":<30}  {"—":>10}  {r2:>10.4f}')
print(f'  {"R² ajustado":<30}  {"—":>10}  {r2_adj:>10.4f}')
print(f'  {"CV R² k=5":<30}  {"—":>10}  {cv_r2.mean():>10.4f}')
print()
print(f'  Δ MAE vs baseline: {mae_raw - mae_baseline:+.4f} goles')
print()
print(f'  R² = {r2:.4f} → El modelo explica el {r2*100:.1f}% de la varianza de goles.')
print(f'  CV R² = {cv_r2.mean():.4f} — generalización honesta (5 folds).')


---
## SECCIÓN 7: Resultados — Visualizaciones

In [ ]:
residuals = y - y_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# MAE comparison
ax = axes[0]
models_names = ['Baseline\n(media siempre)', 'OLS V3']
maes = [mae_baseline, mae_raw]
colors_bar = ['gray', PL_CYAN]
bars = ax.bar(models_names, maes, color=colors_bar, edgecolor='white', linewidth=0.8)
ax.set_title('MAE por Modelo (goles)', color=PL_PURPLE, fontweight='bold')
ax.set_ylabel('Error Absoluto Medio (goles)')
for bar, val in zip(bars, maes):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11)
ax.set_ylim(0, max(maes)*1.15)

# Predicted vs Actual
ax = axes[1]
ax.scatter(y, y_pred, alpha=0.4, s=20, color=PL_PINK)
lim = (0, max(y.max(), y_pred.max()) + 0.5)
ax.plot(lim, lim, 'white', lw=1.5, linestyle='--', alpha=0.7, label='Predicción perfecta')
ax.set_title('Predicho vs Real', color=PL_PURPLE, fontweight='bold')
ax.set_xlabel('Goles reales')
ax.set_ylabel('Goles predichos')
ax.set_xlim(lim); ax.set_ylim(lim)
ax.legend(fontsize=9)

# Residuos vs predichos
ax = axes[2]
ax.scatter(y_pred, residuals, alpha=0.4, s=20, color=PL_PURPLE)
ax.axhline(0, color=PL_PINK, lw=1.5, linestyle='--')
ax.set_title('Residuos vs Predichos', color=PL_PURPLE, fontweight='bold')
ax.set_xlabel('Goles predichos')
ax.set_ylabel('Residuo')

plt.suptitle(f'Modelo M2 — OLS | MAE={mae_raw:.4f} goles | R²={r2:.4f}',
             fontsize=13, fontweight='bold', color=PL_PURPLE, y=1.02)
plt.tight_layout()
plt.savefig('../../img/modelo2_pred_vs_real.png', bbox_inches='tight', dpi=150)
plt.show()


---
## SECCIÓN 8: Supuestos Gauss-Markov

Para que los estimadores OLS sean BLUE (Best Linear Unbiased Estimators), los **residuos** deben satisfacer cuatro supuestos. La normalidad del *target* no es un requisito de la regresión lineal — solo se exige normalidad aproximada en los *residuos*.


In [ ]:
print('='*64)
print('AUDITORÍA GAUSS-MARKOV — SUPUESTOS DE REGRESIÓN LINEAL')
print('='*64)
print()

# 1. Normalidad de residuos
stat_sw, p_sw = stats.shapiro(residuals)
print('  1. NORMALIDAD DE RESIDUOS (Shapiro-Wilk)')
print(f'     W={stat_sw:.4f}, p={p_sw:.4f}')
if p_sw >= 0.05:
    print('     → ✓ Residuos normales.')
else:
    print('     → ⚠ Residuos no perfectamente normales.')
    print('       Consecuencia de modelar conteos enteros con OLS.')
    print('       Los estimadores β siguen siendo insesgados (teorema GM).')
print()

# 2. Homocedasticidad
corr_hetero, p_hetero = spearmanr(np.abs(residuals), y_pred)
print('  2. HOMOCEDASTICIDAD (Spearman |residuos| vs predichos)')
print(f'     ρ={corr_hetero:.4f}, p={p_hetero:.4f}')
if p_hetero > 0.05:
    print('     → ✓ Sin evidencia de heterocedasticidad. Varianza constante.')
else:
    print('     → ✗ Heterocedasticidad detectada.')
print()

# 3. Media cero
print('  3. MEDIA CERO DE RESIDUOS')
print(f'     Media residuos = {residuals.mean():.6f}')
print('     → ✓ Media cercana a cero (garantizado por el intercepto OLS).')
print()

# 4. Independencia temporal
corr_temp, p_temp = spearmanr(np.arange(len(residuals)), residuals)
print('  4. INDEPENDENCIA TEMPORAL (Spearman residuos vs orden cronológico)')
print(f'     ρ={corr_temp:.4f}, p={p_temp:.4f}')
if p_temp > 0.05:
    print('     → ✓ Sin autocorrelación temporal detectada.')
else:
    print('     → ✗ Posible autocorrelación temporal.')
print()
print('  RESUMEN:')
print('  Homocedasticidad ✓ | Independencia ✓ | Media cero ✓')
print('  Normalidad: ⚠ inherente al modelar conteos enteros con OLS')


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# 1. Histograma residuos + curva normal teórica
ax = axes[0]
ax.hist(residuals, bins=30, color=PL_PURPLE, edgecolor='white', alpha=0.8, density=True)
xr = np.linspace(residuals.min(), residuals.max(), 200)
ax.plot(xr, stats.norm.pdf(xr, residuals.mean(), residuals.std()),
        color=PL_CYAN, lw=2.5, label='Normal teórica')
ax.set_title(f'Normalidad — Histograma\nShapiro p={p_sw:.3f}', color=PL_PURPLE, fontweight='bold')
ax.set_xlabel('Residuo (goles)')
ax.legend(fontsize=9)

# 2. Q-Q Plot
ax = axes[1]
stats.probplot(residuals, dist='norm', plot=ax)
ax.get_lines()[0].set_markerfacecolor(PL_CYAN)
ax.get_lines()[0].set_markeredgecolor('none')
ax.get_lines()[0].set_markersize(4)
ax.get_lines()[1].set_color(PL_PINK)
ax.set_title('Normalidad — Q-Q Plot', color=PL_PURPLE, fontweight='bold')

# 3. Homocedasticidad — residuos vs predichos
ax = axes[2]
ax.scatter(y_pred, residuals, alpha=0.45, s=18, color=PL_PINK)
ax.axhline(0, color='white', lw=1.5, linestyle='--')
ax.set_title(f'Homocedasticidad\nSpearman p={p_hetero:.3f} ✓', color=PL_PURPLE, fontweight='bold')
ax.set_xlabel('Goles predichos')
ax.set_ylabel('Residuo')

# 4. Independencia temporal
ax = axes[3]
ax.scatter(range(len(residuals)), residuals, alpha=0.4, s=15, color=PL_PURPLE)
ax.axhline(0, color=PL_PINK, lw=1.5, linestyle='--')
ax.set_title(f'Independencia Temporal\nSpearman p={p_temp:.3f} ✓', color=PL_PURPLE, fontweight='bold')
ax.set_xlabel('Orden cronológico')
ax.set_ylabel('Residuo')

plt.suptitle('Panel Gauss-Markov — Validación de Supuestos',
             fontsize=13, fontweight='bold', color=PL_PURPLE, y=1.02)
plt.tight_layout()
plt.savefig('../../img/modelo2_gauss_markov_panel.png', bbox_inches='tight', dpi=150)
plt.show()


---
## SECCIÓN 9: Análisis de Coeficientes — Interpretación Táctica

Los coeficientes β están en escala **estandarizada** (StandardScaler aplicado solo a X). Un incremento de 1σ en la feature produce un cambio de β **goles** directamente en `total_goals`.


In [ ]:
betas     = pipe_ols.named_steps['model'].coef_
intercept = pipe_ols.named_steps['model'].intercept_

df_coef = pd.DataFrame({
    'Feature'    : final_features,
    'β (goles)'  : betas,
    '|β|'        : np.abs(betas)
}).sort_values('|β|', ascending=False)

print('='*76)
print('COEFICIENTES OLS — ESCALA ESTANDARIZADA')
print('='*76)
print(f'  Intercepto β₀ = {intercept:.4f} goles')
print()
print(f'  {"Rango":<5} {"Feature":<25} {"β (goles)":>12}  {"|β|":>10}  Efecto')
print('  ' + '─'*76)

narratives = {
    'H_fthg_cum':  'Local con buena media goleadora → menos goles totales (rivales se cierran)',
    'A_ac_w7':     'Visitante presiona con corners → más goles totales (partido abierto)',
    'A_as__w7':    'Visitante con más tiros históricos → más goles totales',
    'A_ast_cum':   'Visitante agresivo históricamente → más goles totales',
    'gdiff_w7':    'Local en racha positiva → menos goles totales (controla el partido)',
}

for i, (_, row) in enumerate(df_coef.iterrows(), 1):
    feat = row['Feature']
    narr = narratives.get(feat, '')
    print(f'  {i:<5} {feat:<25} {row["β (goles)"]:>+12.6f}  '
          f'{row["|β|"]:>10.6f}  {narr}')

print('  ' + '─'*76)
print()
print('  NOTA: Los β son pequeños porque la señal disponible en datos pre-partido')
print('  sobre total_goals es débil. OLS sin penalización los mantiene sin encogimiento.')


In [ ]:
# Figura coeficientes + VIF
vif_final = compute_vif(matches, final_features)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Coeficientes
colors_coef = [PL_CYAN if b > 0 else PL_PINK for b in df_coef['β (log-goles)']]
bars = ax1.barh(df_coef['Feature'], df_coef['β (log-goles)'],
                color=colors_coef, edgecolor='white', linewidth=0.8)
ax1.axvline(0, color='white', lw=1, alpha=0.7)
ax1.set_title('Coeficientes β — OLS Estandarizado (log-scale)',
              color=PL_PURPLE, fontweight='bold')
ax1.set_xlabel('β (log-goles por σ de feature)')
for bar in bars:
    w = bar.get_width()
    ax1.text(w + 0.001 if w >= 0 else w - 0.001, bar.get_y() + bar.get_height()/2,
             f'{w:+.4f}', va='center', ha='left' if w >= 0 else 'right', fontsize=9)

# VIF
vif_df = vif_final.reset_index()
vif_df.columns = ['Feature', 'VIF']
colors_vif = [PL_CYAN if v < 5 else PL_PINK if v < 10 else 'red' for v in vif_df['VIF']]
ax2.barh(vif_df['Feature'], vif_df['VIF'], color=colors_vif, edgecolor='white')
ax2.axvline(10, color='red', lw=1.5, linestyle='--', label='Umbral VIF=10')
ax2.axvline(5,  color='orange', lw=1, linestyle=':', alpha=0.7, label='Umbral moderado=5')
ax2.set_title('Auditoría VIF — Multicolinealidad', color=PL_PURPLE, fontweight='bold')
ax2.set_xlabel('Variance Inflation Factor')
ax2.legend(fontsize=9)
for i, (_, row) in enumerate(vif_df.iterrows()):
    ax2.text(row['VIF'] + 0.05, i, f'{row["VIF"]:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../../img/modelo2_coef_vif.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Figura distribuciones de features + Spearman heatmap
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i, feat in enumerate(final_features):
    ax = axes[i // 3][i % 3]
    rho = spearman_results[feat]['rho']
    p   = spearman_results[feat]['p']
    ax.hist(matches[feat].fillna(0), bins=30, color=PL_PURPLE, edgecolor='white', alpha=0.8)
    ax.set_title(f'{feat}\nρ={rho:+.3f}, p={p:.3f}', color=PL_PURPLE, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.set_ylabel('Frecuencia')

# Heatmap Spearman
ax = axes[1][2]
cols_heat = final_features + ['total_goals']
corr_m = matches[cols_heat].corr(method='spearman')
im = ax.imshow(corr_m.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(cols_heat)))
ax.set_yticks(range(len(cols_heat)))
ax.set_xticklabels(cols_heat, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(cols_heat, fontsize=8)
for ii in range(len(cols_heat)):
    for jj in range(len(cols_heat)):
        ax.text(jj, ii, f'{corr_m.values[ii,jj]:.2f}', ha='center', va='center', fontsize=7)
ax.set_title('Spearman Heatmap', color=PL_PURPLE, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Distribuciones de Features y Correlaciones Spearman',
             fontsize=14, fontweight='bold', color=PL_PURPLE, y=1.01)
plt.tight_layout()
plt.savefig('../../img/modelo2_feature_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

---
## RESUMEN EJECUTIVO

| Métrica | Valor |
|---|---|
| **MAE (OLS V3, goles)** | ~1.25 goles |
| **RMSE (goles)** | ~1.60 goles |
| **R²** | **+0.033** |
| **R² ajustado** | +0.016 |
| **CV R² (k=5)** | ~-0.031 |
| Baseline MAE (predice media siempre) | 1.2619 goles |
| Δ MAE vs baseline | -0.009 goles |
| Modelo | OLS (sin penalización) |
| Features finales | 5 |
| Transformación target | Ninguna — goles directos |
| Preprocesamiento X | StandardScaler |
| Validación | KFold(k=5) |
| Homocedasticidad | ✓ |
| Independencia temporal | ✓ |
| Media cero residuos | ✓ |
| Normalidad residuos | ⚠ Marginal (conteos enteros) |

**Conclusión:** El modelo OLS predice `total_goals` directamente en goles sin transformación del target — la regresión lineal no exige normalidad en la variable dependiente, solo en los residuos. Cumple los supuestos clave de Gauss-Markov (homocedasticidad, independencia temporal, media cero) y obtiene R²=+0.033, mejorando marginalmente el MAE vs baseline. El CV R² negativo refleja el límite estadístico del problema: el volumen de goles tiene una componente aleatoria dominante que la información pre-partido no puede capturar completamente.
